# GloVe: Global Vectors for Word Representation

> Jeffrey Pennington,   Richard Socher,   Christopher D. Manning

> https://nlp.stanford.edu/projects/glove/

> https://nlp.stanford.edu/pubs/glove.pdf

> https://www.kaggle.com/hassanamin/glove-based-text-classification <br>
Glove Based Text Classification


### Basics of Using Pre-trained GloVe 

#### Download and unzip embeddings

In [ ]:
!wget --no-check-certificate \
    https://nlp.stanford.edu/data/glove.twitter.27B.zip \
    -O ../data/embeddings/glove.twitter.27B.zip

In [ ]:
import zipfile

local_zip = '../data/embeddings/glove.twitter.27B.zip'

zip_ref = zipfile.ZipFile(local_zip, 'r')

zip_ref.extractall('../data/embeddings')
zip_ref.close()

In [1]:
!head -5 ../data/embeddings/glove.twitter.27B.25d.txt

<user> 0.62415 0.62476 -0.082335 0.20101 -0.13741 -0.11431 0.77909 2.6356 -0.46351 0.57465 -0.024888 -0.015466 -2.9696 -0.49876 0.095034 -0.94879 -0.017336 -0.86349 -1.3348 0.046811 0.36999 -0.57663 -0.48469 0.40078 0.75345
. 0.69586 -1.1469 -0.41797 -0.022311 -0.023801 0.82358 1.2228 1.741 -0.90979 1.3725 0.1153 -0.63906 -3.2252 0.61269 0.33544 -0.57058 -0.50861 -0.16575 -0.98153 -0.8213 0.24333 -0.14482 -0.67877 0.7061 0.40833
: 1.1242 0.054519 -0.037362 0.10046 0.11923 -0.30009 1.0938 2.537 -0.072802 1.0491 1.0931 0.066084 -2.7036 -0.14391 -0.22031 -0.99347 -0.65072 -0.030948 -1.0817 -0.64701 0.32341 -0.41612 -0.5268 -0.047166 0.71549
rt 0.74056 0.9155 -0.16352 0.35843 0.05266 0.1456 1.0421 2.8073 0.12865 1.0492 0.13033 0.20508 -2.6686 -0.50551 -0.29574 -0.91433 -0.40456 -1.0988 -1.0333 -0.17875 0.37979 -0.25922 -0.74854 0.36001 0.61206
, 0.84705 -1.0349 -0.050419 0.27164 -0.58659 0.99514 0.25267 1.6963 0.10313 0.80073 0.74655 -1.2667 -4.036 -0.22557 0.16322 -0.67015 -0.64812 0.0103

### Prepare corpus

In [2]:
import numpy as np

# define documents
docs = ['Well done!',
        'Good work',
        'Great effort',
        'nice work',
        'Excellent!',
        'Weak',
        'Poor effort!',
        'not good',
        'poor work',
        'Could have done better.']
# define class labels
labels = np.array([1,1,1,1,1,0,0,0,0,0])

In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# prepare tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(docs)
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1
print(f"vocab_size: {vocab_size}")

# integer encode the documents
sequences = tokenizer.texts_to_sequences(docs)

vocab_size: 15


In [4]:
word_index

{'work': 1,
 'done': 2,
 'good': 3,
 'effort': 4,
 'poor': 5,
 'well': 6,
 'great': 7,
 'nice': 8,
 'excellent': 9,
 'weak': 10,
 'not': 11,
 'could': 12,
 'have': 13,
 'better': 14}

In [5]:
sequences

[[6, 2],
 [3, 1],
 [7, 4],
 [8, 1],
 [9],
 [10],
 [5, 4],
 [11, 3],
 [5, 1],
 [12, 13, 2, 14]]

In [6]:
max_length = 4
padded_docs = pad_sequences(sequences, maxlen=max_length, padding='post')

In [7]:
padded_docs

array([[ 6,  2,  0,  0],
       [ 3,  1,  0,  0],
       [ 7,  4,  0,  0],
       [ 8,  1,  0,  0],
       [ 9,  0,  0,  0],
       [10,  0,  0,  0],
       [ 5,  4,  0,  0],
       [11,  3,  0,  0],
       [ 5,  1,  0,  0],
       [12, 13,  2, 14]], dtype=int32)

### Trained Embedings

In [8]:
import tensorflow as tf
print(tf.__version__)

embedding_dim = 25

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(6, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

2.4.1
Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding (Embedding)        (None, 4, 25)             375       
_________________________________________________________________
flatten (Flatten)            (None, 100)               0         
_________________________________________________________________
dense (Dense)                (None, 6)                 606       
_________________________________________________________________
dense_1 (Dense)              (None, 1)                 7         
Total params: 988
Trainable params: 988
Non-trainable params: 0
_________________________________________________________________


2021-07-06 16:32:37.433482: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2021-07-06 16:32:37.433719: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [9]:
num_epochs = 10
model.fit(padded_docs, labels, epochs=num_epochs)

2021-07-06 16:32:40.534647: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:116] None of the MLIR optimization passes are enabled (registered 2)


Epoch 1/10
1/1 [==============================] - 0s 384ms/step - loss: 0.6908 - accuracy: 0.5000
Epoch 2/10
1/1 [==============================] - 0s 2ms/step - loss: 0.6878 - accuracy: 0.5000
Epoch 3/10
1/1 [==============================] - 0s 2ms/step - loss: 0.6851 - accuracy: 0.5000
Epoch 4/10
1/1 [==============================] - 0s 2ms/step - loss: 0.6824 - accuracy: 0.6000
Epoch 5/10
1/1 [==============================] - 0s 2ms/step - loss: 0.6799 - accuracy: 0.6000
Epoch 6/10
1/1 [==============================] - 0s 2ms/step - loss: 0.6771 - accuracy: 0.6000
Epoch 7/10
1/1 [==============================] - 0s 2ms/step - loss: 0.6746 - accuracy: 0.7000
Epoch 8/10
1/1 [==============================] - 0s 3ms/step - loss: 0.6719 - accuracy: 0.6000
Epoch 9/10
1/1 [==============================] - 0s 2ms/step - loss: 0.6693 - accuracy: 0.7000
Epoch 10/10
1/1 [==============================] - 0s 1ms/step - loss: 0.6667 - accuracy: 0.7000


In [10]:
e = model.layers[0]
weights = e.get_weights()[0]
print(weights.shape) # shape: (vocab_size, embedding_dim)

(15, 25)


In [11]:
weights

array([[-0.0155776 ,  0.0011588 ,  0.01755733,  0.02365235, -0.01684001,
         0.04112972, -0.03526505, -0.00831218, -0.03719359,  0.02617737,
         0.01581288,  0.05055582,  0.00911763, -0.00600285,  0.04535009,
        -0.00179326, -0.03768042,  0.00670663,  0.03626745,  0.01078076,
         0.04669245,  0.03772639, -0.0283434 , -0.03800798, -0.00307831],
       [ 0.05686874,  0.01326092, -0.04740536, -0.02393729, -0.051433  ,
         0.00387287,  0.01624452,  0.02287052,  0.0520949 ,  0.0541614 ,
        -0.05014268, -0.05787386,  0.0058808 ,  0.02816481, -0.01385946,
         0.00252028, -0.00125009,  0.01592473,  0.03739512, -0.03987742,
        -0.02368581, -0.00191334, -0.0188308 ,  0.01247564,  0.01649545],
       [ 0.00072686,  0.04852612, -0.0528    , -0.02362676,  0.00790729,
         0.04868065,  0.00851007,  0.05539016, -0.00781499,  0.03140577,
        -0.05564618, -0.01118929,  0.03485227,  0.03300466,  0.0039037 ,
         0.049018  ,  0.00118474, -0.02910737,  0

### PRE-Trained Embedings

In [13]:
glove_embedding_path = "../data/embeddings/glove.twitter.27B.25d.txt"

In [14]:
# Load all words and vectors to dictionary
embeding_dim = 25
glove_embeddings_dict = {}
with open(glove_embedding_path, 'r', encoding="utf-8") as embeddings:
    for line in embeddings:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], "float32")
        glove_embeddings_dict[word] = vector

print(f"Num of words: {len(glove_embeddings_dict.keys())}")

Num of words: 1193514


In [15]:
import random

for word in random.sample(glove_embeddings_dict.keys(), 10):
    print(word)
    print(glove_embeddings_dict[word])
    print(f"SHAPE: {glove_embeddings_dict[word].shape}")


جودو
[ 0.42545   1.7504   -1.0527    0.66677   0.49765  -0.90738  -0.37077
  0.71041   1.7829    1.1483   -0.80608   0.88564   1.9       0.64013
  0.98997   1.2862    1.7314   -2.3778    0.83961   0.050871 -0.56085
 -0.33455  -0.16737   0.21086  -1.2957  ]
SHAPE: (25,)
legalidade
[ 0.31976   0.51029   0.47697   0.98551   0.78165  -1.2393   -1.5552
 -1.2699    0.9433   -0.55029  -1.38     -1.1662   -0.10728   0.59883
 -0.12388   0.45671  -0.29403   1.3152    0.8712    0.30639   0.097755
  1.6716   -0.14565  -1.0405   -0.34195 ]
SHAPE: (25,)
fangirls
[ 0.70908  -0.11759   0.40154  -0.39432  -0.14465  -0.53858   1.1101
 -0.44316  -0.92368  -0.4695   -0.38187   0.3576   -1.3512   -0.41925
 -0.28028   1.3436    0.078465 -0.53631   1.1247   -0.026117  0.27585
  0.79378  -0.6414   -0.85992  -0.7111  ]
SHAPE: (25,)
ковалев
[-0.18553  -0.45919  -0.004735 -1.2249   -0.83605  -1.5389   -0.26455
 -2.1572    0.40918   0.12678   0.3906    1.6634    0.32578  -0.459
  0.3845   -0.48544  -0.13333   1.7

In [16]:
print(f"work: {glove_embeddings_dict['work']}")
print(f"bad: {glove_embeddings_dict['bad']}")
print(f"python: {glove_embeddings_dict['python']}")

work: [-0.43103   0.90215  -0.18112  -0.53684   0.39468   0.14496   1.3358
 -0.93469   0.057636 -0.016199 -0.89613   0.1223   -5.3107    0.085847
  0.73645  -0.010881  0.53315  -0.62596  -0.19967  -0.82147  -0.34758
 -0.34909   0.20439   0.81419   0.16562 ]
bad: [ 0.41388   0.022308  0.056536 -0.010503  0.27395   0.71342   1.6414
 -0.11188  -0.26249   0.10783  -0.95191   0.40414  -4.6303   -0.18866
  0.60344   0.24891   0.36794  -0.50313  -0.48655  -0.21142   0.38759
  1.0411    0.42695   0.2866    0.02321 ]
python: [-0.25645  -0.22323   0.025901  0.22901   0.49028  -0.060829  0.24563
 -0.84854   1.5882   -0.7274    0.60603   0.25205  -1.8064   -0.95526
  0.44867   0.013614  0.60856   0.65423   0.82506   0.99459  -0.29403
 -0.27013  -0.348    -0.7293    0.2201  ]


#### Create embedding matrix

In [18]:
word_index.items()

dict_items([('work', 1), ('done', 2), ('good', 3), ('effort', 4), ('poor', 5), ('well', 6), ('great', 7), ('nice', 8), ('excellent', 9), ('weak', 10), ('not', 11), ('could', 12), ('have', 13), ('better', 14)])

In [36]:
glove_embedding_matrix = np.zeros((vocab_size, embeding_dim))

for word, i in tokenizer.word_index.items():
    glove_embedding_vector = glove_embeddings_dict.get(word)
    if glove_embedding_vector is not None:
        glove_embedding_matrix[i] = glove_embedding_vector

In [21]:
glove_embedding_matrix

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [-4.31030005e-01,  9.02149975e-01, -1.81119993e-01,
        -5.36840022e-01,  3.94679993e-01,  1.44960001e-01,
         1.33580005e+00, -9.34689999e-01,  5.76360002e-02,
        -1.61990002e-02, -8.96130025e-01,  1.22299999e-01,
        -5.31069994e+00,  8.58469978e-02,  7.36450016e-01,
        -1.08810002e-02,  5.33150017e-01, -6.25959992e-01,
        -1.99670002e-01, -8.21470022e-01, -3.47579986e-01,
        -3.49090010e-01,  2.04390004e-01,  8.14189970e-01,
         1.65619999e-01],
    

In [33]:
embedding_dim = 25

glove_model = tf.keras.Sequential([
    # tf.keras.layers.Embedding(vocab_size, embedding_dim, weights=[glove_embedding_matrix], input_length=max_length, trainable=False),
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length, 
                              embeddings_initializer=tf.keras.initializers.Constant(glove_embedding_matrix), trainable=False),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(6, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
glove_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
glove_model.summary()

Model: "sequential_5"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_5 (Embedding)      (None, 4, 25)             375       
_________________________________________________________________
flatten_5 (Flatten)          (None, 100)               0         
_________________________________________________________________
dense_10 (Dense)             (None, 6)                 606       
_________________________________________________________________
dense_11 (Dense)             (None, 1)                 7         
Total params: 988
Trainable params: 613
Non-trainable params: 375
_________________________________________________________________


In [34]:
num_epochs = 10
glove_model.fit(padded_docs, labels, epochs=num_epochs)

Epoch 1/10
1/1 [==============================] - 0s 310ms/step - loss: 1.0334 - accuracy: 0.5000
Epoch 2/10
1/1 [==============================] - 0s 2ms/step - loss: 1.0154 - accuracy: 0.5000
Epoch 3/10
1/1 [==============================] - 0s 3ms/step - loss: 0.9979 - accuracy: 0.5000
Epoch 4/10
1/1 [==============================] - 0s 2ms/step - loss: 0.9847 - accuracy: 0.5000
Epoch 5/10
1/1 [==============================] - 0s 2ms/step - loss: 0.9718 - accuracy: 0.5000
Epoch 6/10
1/1 [==============================] - 0s 2ms/step - loss: 0.9592 - accuracy: 0.5000
Epoch 7/10
1/1 [==============================] - 0s 3ms/step - loss: 0.9467 - accuracy: 0.5000
Epoch 8/10
1/1 [==============================] - 0s 2ms/step - loss: 0.9345 - accuracy: 0.5000
Epoch 9/10
1/1 [==============================] - 0s 2ms/step - loss: 0.9224 - accuracy: 0.5000
Epoch 10/10
1/1 [==============================] - 0s 1ms/step - loss: 0.9106 - accuracy: 0.5000


In [24]:
glove_e = glove_model.layers[0]
glove_weights = glove_e.get_weights()[0]
print(glove_weights.shape) 

(15, 25)


In [25]:
glove_weights

array([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00],
       [-4.3103e-01,  9.0215e-01, -1.8112e-01, -5.3684e-01,  3.9468e-01,
         1.4496e-01,  1.3358e+00, -9.3469e-01,  5.7636e-02, -1.6199e-02,
        -8.9613e-01,  1.2230e-01, -5.3107e+00,  8.5847e-02,  7.3645e-01,
        -1.0881e-02,  5.3315e-01, -6.2596e-01, -1.9967e-01, -8.2147e-01,
        -3.4758e-01, -3.4909e-01,  2.0439e-01,  8.1419e-01,  1.6562e-01],
       [-2.6536e-01,  1.5467e+00,  3.9805e-01, -5.5764e-01, -5.1273e-02,
        -9.0431e-02,  1.2663e+00, -1.4903e-01, -7.2469e-01,  3.9007e-01,
        -2.5664e-01,  8.7859e-01, -4.1568e+00, -2.4098e-01,  5.3971e-01,
        -4.2647e-01,  5.6943e-01, -6.4409e-01, -8

### How to handle OOV?

In [39]:
for i in glove_embeddings_dict.keys():
    if "oov" in i:
        print(i)

oovoo
groove
loooove
looove
looooove
hoover
loove
groovy
loooooove
smoove
looooooove
grooves
loooooooove
groovin
hooves
grooving
hoovering
loovee
smoov
grooveshark
looooooooove
oovoome
oovoo.try
looovvve
noovo
looovee
loooooooooove
looovve
loovve
loooovvve
poovo
looooooooooove
moov
proove
moove
hoovered
loovvee
oovooing
hoovers
moooove
looooved
loooooooooooove
loooovve
mooove
oovo
loooovee
looooves
loooved
loovvve
loooovvvve
loooves
moovie
looooovvve
loooooved
looooooooooooove
looovvee
moovil
nooovo
mooooove
loooooooooooooove
groovemusicapp
looovvvve
looooovvvve
waterlooville
loooooves
sooverit
grooved
noova
proova
looooooooooovvvvvve
looooooooooooooove
oovoo.ntry
looves
pooovo
groover
oov
hooverphonic
hoova
zoovie
joovy
looooving
groovers
noooovo
oover
looooovve
moooooove
noovio
loovens
groovey
looooooved
loooooooooooooooove
looovvvee
loov
denoovo
loooooovvve
groov
looooovee
smoovefollowtrain
loooovvee
loooovely
noovela
mooovil
looved
looooooves
loooooovvvve
springroove
looooovvvvve
n